### SAX alphabet size & PAA word size (FARSEEING CV)

In [1]:
%load_ext autoreload
%autoreload 2
import warnings
warnings.simplefilter(action='ignore')

from src.costream.data import group_files_by_subject
from src.costream.evaluation import run_subject_cv, ModelSpec
import numpy as np
import pandas as pd
from src.fall_lm import FallLM
from src import utils

FALL_DIR = "data/farseeing/fall"
FREQ = 100
WINDOW_SEC = 3

In [2]:
data_desc = pd.read_excel("data/farseeing/description.xlsx", engine='openpyxl',
                          usecols=['Randomnumber', 'Sensor_location', 'Sample_rate_Hz'])
l5 = data_desc[data_desc['Sensor_location'] == 'L5']
l5 = l5[l5['Sample_rate_Hz'] == FREQ]
l5_subjects = set(l5['Randomnumber'].str.split('-').str[0])

fall_subject_map = group_files_by_subject(FALL_DIR, id_extractor=utils.fall_id_extractor)
fall_subject_map = {s: fps for s, fps in fall_subject_map.items() if s in l5_subjects}
print(f"Found {len(fall_subject_map)} fall subjects")

Found 40 fall subjects


In [3]:
subject_df_map = {s: utils.load_fall_subject(fall_subject_map[s]) for s in fall_subject_map.keys()}

In [4]:
BASE_KW = dict(vectorizer_type="tfidf", alpha=1.5, use_diff=False, impact_mode="category", random_state=42)
alphabet_specs = [
    ModelSpec(f"n_bins={n}", FallLM(n_bins=n, word_size=3, ngram_range=(4, 5), **BASE_KW))
    for n in range(3, 11)
]
wordsize_specs = [
    ModelSpec(f"word_size={w}", FallLM(n_bins=5, word_size=w, ngram_range=(4, 5), **BASE_KW))
    for w in range(3, 31, 3)
]

specs = alphabet_specs + wordsize_specs
len(specs)

18

In [5]:
import os
os.makedirs('output/results', exist_ok=True)

cv_res = run_subject_cv(
    subject_map=subject_df_map,
    model_specs=specs,
    feature_cols=['mag'],
    window_size=WINDOW_SEC,
    cv=5,
    step=1.0,
    spacing="multiphase",
    verbose=True,
    use_post_event_data=False,
    tolerance=[3, 20],
)
cv_res.to_csv('output/results/cv_results_ablation.csv', index=False)


=== Fold 1/5 | Train Subjects: 32 | Test Subjects: 8 ===
  Segmented Train Data: (1627, 300)

=== Fold 2/5 | Train Subjects: 32 | Test Subjects: 8 ===
  Segmented Train Data: (2563, 300)

=== Fold 3/5 | Train Subjects: 32 | Test Subjects: 8 ===
  Segmented Train Data: (2380, 300)

=== Fold 4/5 | Train Subjects: 32 | Test Subjects: 8 ===
  Segmented Train Data: (2647, 300)

=== Fold 5/5 | Train Subjects: 32 | Test Subjects: 8 ===
  Segmented Train Data: (2755, 300)


In [6]:
metrics = ["f1-score", "precision", "recall", "false alarm rate"]

def agg_table(df, models, key, key_name):
    sub = df[df["model"].isin(models)].copy()
    sub[key_name] = sub["model"].map(key)
    g = sub.groupby(key_name)
    out = g[metrics].mean().round(3)
    for m in metrics:
        out[m + "_std"] = g[m].std().round(3)
    return out.reset_index().sort_values(key_name)

In [7]:
alphabet_models = [s.name for s in alphabet_specs]
alphabet_key = {s.name: int(s.name.split("=")[1]) for s in alphabet_specs}
alphabet_table = agg_table(cv_res, alphabet_models, alphabet_key, "n_bins")
alphabet_table.to_csv("output/results/ablation_alphabet.csv", index=False)
alphabet_table.to_latex("output/results/ablation_alphabet.tex", index=False, float_format="%.3f")
alphabet_table

,n_bins,f1-score,precision,recall,false alarm rate,f1-score_std,precision_std,recall_std,false alarm rate_std
0,3,0.645,0.517,0.882,0.291,0.106,0.125,0.083,0.147
1,4,0.518,0.423,0.775,0.493,0.175,0.225,0.069,0.391
2,5,0.642,0.518,0.866,0.283,0.092,0.106,0.100,0.148
3,6,0.618,0.514,0.822,0.317,0.142,0.182,0.055,0.261
4,7,0.637,0.528,0.830,0.278,0.125,0.146,0.075,0.196
5,8,0.597,0.484,0.826,0.345,0.136,0.171,0.067,0.245
6,9,0.626,0.505,0.860,0.327,0.145,0.160,0.042,0.233
7,10,0.633,0.519,0.836,0.277,0.104,0.131,0.070,0.145


In [8]:
wordsize_models = [s.name for s in wordsize_specs]
wordsize_key = {s.name: int(s.name.split("=")[1]) for s in wordsize_specs}
wordsize_table = agg_table(cv_res, wordsize_models, wordsize_key, "word_size")
wordsize_table.to_csv("output/results/ablation_wordsize.csv", index=False)
wordsize_table.to_latex("output/results/ablation_wordsize.tex", index=False, float_format="%.3f")
wordsize_table

,word_size,f1-score,precision,recall,false alarm rate,f1-score_std,precision_std,recall_std,false alarm rate_std
0,3,0.642,0.518,0.866,0.283,0.092,0.106,0.100,0.148
1,6,0.614,0.493,0.841,0.320,0.122,0.130,0.092,0.203
2,9,0.611,0.492,0.840,0.314,0.106,0.128,0.081,0.169
3,12,0.587,0.466,0.809,0.313,0.111,0.116,0.120,0.102
4,15,0.577,0.454,0.796,0.317,0.112,0.104,0.121,0.086
5,18,0.588,0.461,0.815,0.310,0.091,0.080,0.116,0.059
6,21,0.538,0.416,0.771,0.361,0.087,0.091,0.056,0.088
7,24,0.536,0.405,0.802,0.389,0.070,0.069,0.062,0.088
8,27,0.526,0.398,0.786,0.393,0.083,0.078,0.109,0.095
9,30,0.503,0.379,0.759,0.408,0.051,0.050,0.101,0.090


### SAX alphabet size ablation on FallAllD (subject-wise 5-fold CV)

In [9]:
from src.fallalld import load_fallalld, adapter_fallalld_to_costream

FALLALLD_ORIG_FREQ = 238

fallalld_df = load_fallalld()
fallalld_subject_map_raw = adapter_fallalld_to_costream(fallalld_df)

fallalld_subject_map = {
    subj: [utils.resample_df(rec, ['mag'], FALLALLD_ORIG_FREQ, FREQ) for rec in recs]
    for subj, recs in fallalld_subject_map_raw.items()
}
print(f"{len(fallalld_subject_map)} FallAllD subjects")

Adapting DataFrame to Costream format...


100%|██████████| 1745/1745 [00:00<00:00, 8442.14it/s]


14 FallAllD subjects


In [10]:
cv_res_fallalld = run_subject_cv(
    subject_map=fallalld_subject_map,
    model_specs=alphabet_specs,
    feature_cols=['mag'],
    window_size=WINDOW_SEC,
    cv=5,
    step=1.0,
    spacing="multiphase",
    verbose=True,
    use_post_event_data=False,
    tolerance=[3, 20],
)
cv_res_fallalld.to_csv('output/results/cv_results_ablation_fallalld.csv', index=False)


=== Fold 1/5 | Train Subjects: 11 | Test Subjects: 3 ===
  Segmented Train Data: (6829, 300)

=== Fold 2/5 | Train Subjects: 11 | Test Subjects: 3 ===
  Segmented Train Data: (5391, 300)

=== Fold 3/5 | Train Subjects: 11 | Test Subjects: 3 ===
  Segmented Train Data: (5795, 300)

=== Fold 4/5 | Train Subjects: 11 | Test Subjects: 3 ===
  Segmented Train Data: (4860, 300)

=== Fold 5/5 | Train Subjects: 12 | Test Subjects: 2 ===
  Segmented Train Data: (5997, 300)


In [11]:
alphabet_table_fallalld = agg_table(cv_res_fallalld, alphabet_models, alphabet_key, "n_bins")
alphabet_table_fallalld.to_csv("output/results/ablation_alphabet_fallalld.csv", index=False)
alphabet_table_fallalld.to_latex("output/results/ablation_alphabet_fallalld.tex", index=False, float_format="%.3f")
alphabet_table_fallalld

,n_bins,f1-score,precision,recall,false alarm rate,f1-score_std,precision_std,recall_std,false alarm rate_std
0,3,0.745,0.607,0.971,2.918,0.047,0.061,0.028,0.733
1,4,0.753,0.616,0.973,2.824,0.046,0.061,0.030,0.734
2,5,0.746,0.606,0.976,2.953,0.047,0.062,0.026,0.769
3,6,0.751,0.615,0.969,2.824,0.048,0.062,0.030,0.734
4,7,0.727,0.631,0.887,2.503,0.071,0.061,0.173,1.053
5,8,0.747,0.615,0.956,2.779,0.046,0.059,0.020,0.703
6,9,0.738,0.666,0.828,1.935,0.064,0.074,0.049,0.565
7,10,0.741,0.613,0.943,2.779,0.050,0.064,0.036,0.751


### Impact-token configuration ablation (FallAllD → FARSEEING transfer)

In [12]:
from src.costream.evaluation import train_models, evaluate_models
from src.costream.data.utils import extract_streaming_data
from sklearn.model_selection import KFold, train_test_split

# Mirror experiments.ipynb: subject-wise 80/20 split of FallAllD (same seed → same split)
fallalld_subjects = list(fallalld_subject_map_raw.keys())
fallalld_train_subjects, fallalld_test_subjects = train_test_split(
    fallalld_subjects, test_size=0.2, random_state=42)

fallalld_train_dfs = [df for s in fallalld_train_subjects for df in fallalld_subject_map_raw[s]]
fallalld_test_dfs  = [df for s in fallalld_test_subjects  for df in fallalld_subject_map_raw[s]]

# segment_and_save resamples from 238→100 Hz and applies the current pipeline (incl. oversampling)
X_train_f, y_train_f, test_signals_f, test_events_f = utils.segment_and_save(
    fallalld_train_dfs, fallalld_test_dfs,
    feat_cols=['mag'], win_sec=WINDOW_SEC, thresh=1.4,
    suffix="fallalld", freq=FREQ, orig_freq=FALLALLD_ORIG_FREQ,
)

Resampling from 238Hz to 100Hz...
WIN_SEC=3:
  Train: X=(5597, 300), y=(5597,), balance=[5242  355] (0=ADL, 1=Fall)
  Test:  380 signals, 380 event lists


In [13]:
# Reproduce the same 5-fold FARSEEING test partition as experiments.ipynb
fall_subjects = list(subject_df_map.keys())
kf = KFold(n_splits=5, shuffle=True, random_state=42)
farseeing_test_splits_impact = []
for fold_idx, (_, test_idx) in enumerate(kf.split(fall_subjects)):
    test_subjs = [fall_subjects[i] for i in test_idx]
    fold_signals, fold_events = extract_streaming_data(
        subject_map=subject_df_map,
        subjects=test_subjs,
        feature_col="mag",
        label_col="label",
    )
    farseeing_test_splits_impact.append((fold_signals, fold_events))
    print(f"Fold {fold_idx}: {len(test_subjs)} subjects, {len(fold_signals)} recordings")

Fold 0: 8 subjects, 34 recordings
Fold 1: 8 subjects, 34 recordings
Fold 2: 8 subjects, 29 recordings
Fold 3: 8 subjects, 24 recordings
Fold 4: 8 subjects, 26 recordings


In [14]:
# Production hyperparameters; only use_impact / impact_mode vary across the three configs
BASE_IMPACT_KW = dict(
    n_bins=5, word_size=3, ngram_range=(4, 5),
    vectorizer_type="tfidf", alpha=1.5, use_diff=False, random_state=42,
)

impact_specs = [
    ModelSpec("SAX only", FallLM(**BASE_IMPACT_KW, use_impact=False, use_rel_impact=False)),
    ModelSpec("SAX + impact (3-level)", FallLM(**BASE_IMPACT_KW, use_impact=True, use_rel_impact=True, impact_mode="category")),
    ModelSpec("SAX + impact (12-bin)", FallLM(**BASE_IMPACT_KW, use_impact=True, use_rel_impact=True, impact_mode="physical")),
]

In [15]:
# Train once on FallAllD; evaluate on FallAllD held-out test set
trained_impact = train_models(X_train_f, y_train_f, impact_specs, verbose=True)

res_impact_fallalld = evaluate_models(
    trained_impact,
    test_signals=test_signals_f,
    test_event_points=test_events_f,
    window_size=3.0, step=1.0, freq=100, signal_thresh=1.4,
    tolerance=[3, 20], debounce_secs=60, verbose=False,
)
res_impact_fallalld.to_csv("output/results/ablation_impact_fallalld.csv", index=False)
display(res_impact_fallalld[["model", "f1-score", "precision", "recall"]])

Resampling training data to address class imbalance...
TRAINING 3 models...


,model,f1-score,precision,recall
0,SAX only,0.670213,0.807692,0.572727
1,SAX + impact (3-level),0.737931,0.594444,0.972727
2,SAX + impact (12-bin),0.793893,0.684211,0.945455


In [16]:
# Evaluate the same trained models on each of the 5 FARSEEING test folds
all_farseeing_impact = []
for split_idx, (split_signals, split_events) in enumerate(farseeing_test_splits_impact):
    res = evaluate_models(
        trained_impact,
        test_signals=split_signals,
        test_event_points=split_events,
        window_size=3.0, step=1.0, freq=100, signal_thresh=1.4,
        tolerance=[3, 20], debounce_secs=60, verbose=False,
    )
    res["split"] = split_idx
    all_farseeing_impact.append(res)

res_impact_farseeing = pd.concat(all_farseeing_impact, ignore_index=True)
res_impact_farseeing.to_csv("output/results/ablation_impact_farseeing.csv", index=False)

In [17]:
order = ["SAX only", "SAX + impact (3-level)", "SAX + impact (12-bin)"]
impact_metrics = ["f1-score", "precision", "recall"]

farseeing_stats = res_impact_farseeing.groupby("model")[impact_metrics].agg(["mean", "std"])
fa_idx = res_impact_fallalld.set_index("model")

rows = []
for m in order:
    row = {"Configuration": m}
    for met in impact_metrics:
        row[f"FallAllD {met}"] = f"{fa_idx.loc[m, met]:.3f}"
    for met in impact_metrics:
        mean = farseeing_stats.loc[m, (met, "mean")]
        std  = farseeing_stats.loc[m, (met, "std")]
        row[f"FARSEEING {met}"] = f"{mean:.3f} ({std:.3f})"
    rows.append(row)

impact_table = pd.DataFrame(rows).set_index("Configuration")
impact_table.to_csv("output/results/ablation_impact_token.csv")
impact_table.to_latex("output/results/ablation_impact_token.tex")
impact_table

,FallAllD f1-score,FallAllD precision,FallAllD recall,FARSEEING f1-score,FARSEEING precision,FARSEEING recall
Configuration,,,,,,
SAX only,0.670,0.808,0.573,0.418 (0.057),0.302 (0.048),0.692 (0.108)
SAX + impact (3-level),0.738,0.594,0.973,0.667 (0.093),0.682 (0.136),0.668 (0.111)
SAX + impact (12-bin),0.794,0.684,0.945,0.464 (0.136),0.711 (0.184),0.350 (0.119)
